In [ ]:
from astropy.table import Table
from astropy.io import fits
import glob
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
from tqdm import tqdm
from tsfilt import BilateralFilter

# DEVICE
# Note: This part allows the code to run on GPU if available (and alsp MPS for Apple Silicon Model), otherwise it will run on CPU.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("DEVICE:", DEVICE)

#------------------------------

PATH_TO_DATA = "../Data/"
LCs = sorted(glob.glob(PATH_TO_DATA+"*.fits"))
LCs_filenames = [os.path.basename(x) for x in LCs]

PATH_TO_GROUND_TRUTH = "../Data/"
GTs = [PATH_TO_GROUND_TRUTH + "resp_"+"_".join(f.split(".")[0].split("_")[1:3])+".fits" for f in LCs_filenames]

def read_fits_file(file, bin=False, noisemode="GaussianNoise", snr=None, lam=None):
    '''
    Reads a fits file and returns a pandas dataframe with the time series data.
    It also adds noise to the data if specified and bins the data if specified.
    '''
    fits_file = fits.open(file)
    df = Table(fits_file[1].data).to_pandas()
    df = df.interpolate(method='linear', axis=0)

    attr = df.keys()
    attr = attr[2:]
    if noisemode == "GaussianNoise": 

        if snr not in [None, False]:
            try:
                for k in attr:
                    '''
                    Add noise based on Sacchi's lecture
                    Noise contains Gaussian noise with \sigma = \sqrt{Es/(snr*En)}
                    '''
                    noise = np.random.randn(len(df))
                    Es = np.sum(df[k]**2)
                    En = np.sum(noise**2)
                    alpha = np.sqrt(Es/(snr*En))
                    df[k] = np.maximum(df[k] + alpha * noise, 0)
            except:
                raise Exception("SNR is not specified. Please specify SNR or set it to None for noise free solver.")

    elif noisemode == None:
        pass

    else:
        raise Exception("Mode not found. Please specify whether the solver will be as noise free or not.")

    if bin != False:
        if type(bin) in [float, int]:     
            df = binning(df,bin_size=float(bin))
        else:
            raise ValueError("Bin Size must be specified! and it must be float or int.")
    return df

def binning(df, bin_size):
    """
    Binning the time series data into bins of size bin_size (in days)
    i.e. 0.5 TU means 2 observations per TU
         1 TU means 1 observation per TU
         2 TU means 1 observation for every 2 TU
    """
    df['bin'] = df['time'] // bin_size
    df = df.groupby('bin').mean()
    df['time'] = df.index * bin_size
    return df

def load_model(model_name,model,optimizer,scheduler):
    ''' Loads the model, optimizer, scheduler, epoch and loss history from a checkpoint file. 
    The checkpoint file is expected to be in the "Trained_Model" directory and named as "{model_name}.pt".  
    The checkpoint file should contain the following keys:
        - 'model_state_dict': The state dictionary of the model.
        - 'optimizer_state_dict': The state dictionary of the optimizer.
        - 'scheduler_state_dict': The state dictionary of the scheduler.
        - 'epoch': The epoch number at which the checkpoint was saved.
        - 'loss': The loss history up to the epoch at which the checkpoint was saved.
    '''
    checkpoint = torch.load(f"Trained_Model/{model_name}.pt")
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    epoch = checkpoint['epoch']
    loss_history = checkpoint['loss']
    return model, optimizer, scheduler, epoch, loss_history

In [ ]:
bin_size = 5.0
snr = None
noisemode = "GaussianNoise"

def preprocessing(lightcurve, mode, noisemode=None, snr=None, filter = False, n_iteration=1, n_filters=6):
    '''
    Preprocessing the lightcurve data by applying filtering and binning/sampling
    1. read the fits file
    2. apply filtering (if specified)
    3. apply binning/sampling (if specified)
    4. return the preprocessed dataframe
    '''
    df = read_fits_file(lightcurve, noisemode=noisemode, snr=snr)
    if filter == True:
        for k in df.keys()[2:]:
            sigma_i_prefactor = 4.0
            sigma_x = 3.0
            for _ in range(n_filters):
                filt = BilateralFilter(win_size=int(4*sigma_x+1), sigma_d=sigma_x, sigma_i= sigma_i_prefactor*np.nanstd(df[k]),n_iter=n_iteration)
                df[k] = filt.fit_transform(df[k].to_numpy())

                sigma_i_prefactor -= (sigma_i_prefactor -1)/n_filters
                sigma_x += 9.0

    if mode == 'binning':
        df = binning(df, bin_size)
    elif mode == 'sampling':
        df = df[::bin_size]
    elif mode == "Nobin":
        pass
    else:
        raise ValueError("Mode not recognized")
    return df

# If the filtered data is already saved, load it. Otherwise, preprocess and save it.
load_data = False
if not load_data:
    df = preprocessing(LCs[0], mode='binning', filter=True, snr=snr, noisemode=noisemode)
    T = df['time'].to_numpy()
    h = df['lc_h'].to_numpy()
    s = df['lc_s'].to_numpy()

    out_filt_file = LCs[0].split("/")[-1].replace(".fits","")
    df_filtered = pd.DataFrame({"Time" : T.tolist(),"h":h.tolist(), "s":s.tolist()})
    os.makedirs("Filtered_LCs/",exist_ok=True)
    df_filtered.to_csv(f"Filtered_LCs/{out_filt_file}.csv")

else:
    out_filt_file = LCs[0].split("/")[-1].replace(".fits","")
    df = pd.read_csv(f"Filtered_LCs/{out_filt_file}.csv")
    T = df['Time'].to_numpy()
    h = df['h'].to_numpy()
    s = df['s'].to_numpy()

### 1 Process Reverberation Model

In [ ]:
def solve_for_driving_signal(h, s, bh, bs):
    '''
    Solve for the driving signal 'a' given the hard and soft lightcurves (h,s) and the transfer function parameters (bh, bs).
    Negative driving signals is allowed up to 30% of overall signal.
    -----------
    h : array-like
        Hard lightcurve
    s : array-like
        Soft lightcurve
    bh : float
        Hard transfer function width
    bs : float
        Soft transfer function width
    -----------
    returns:
    a : array-like
        Driving signal
    Rs : float
        Soft transfer function scale
    Rh : float
        Hard transfer function scale
    ----------- 
    '''
    
    bh_0 = 19./30.
    bs_0 = 7./3.
    Rs_0 = 1.0
    Rh_0 = 0.5
    alpha_0 = bs_0/bh_0

    Rs = 1.00 * (bs_0 + Rs_0) - bs
    Rh = 1.00 * (bh_0 + Rh_0/alpha_0) - bh

    assert Rs > 0 and Rh > 0, "Combination of parameters invalid"

    numerator = s/Rs - h/Rh
    denominator = bs/Rs - bh/Rh
    a = numerator/denominator

    tolerance = -0.5
    
    if sum(a < tolerance) > 0.3 * len(s):
        raise ValueError("Combination of parameters invalid")    
    return a, Rs, Rh

In [ ]:
class REV_Model(nn.Module):
    def __init__(self, h, s, bs=19.0/30.0, bh=7.0/3.0):
        '''
        Docstring for __init__
        -----------
        h : array-like
            Hard lightcurve
        s : array-like
            Soft lightcurve
        bs : float
            Soft transfer function width
        bh : float
            Hard transfer function width
        -----------
        Kernel is defined as first guess of the transfer function, which is a top-hat function of width 2000 s.
        Driving signal 'a' is solved using the provided h and s lightcurves and the transfer function parameters.
        -----------
        '''
        super().__init__()
        
        assert len(h) == len(s), "Length of h and s must be equal"
        
        self.N = len(h)
        self.bh = torch.tensor(bh)     # Default: 0.63
        self.bs = torch.tensor(bs)     # Default: 2.33

        self.Kernel = nn.Parameter(torch.tensor([1.0/2000.0 if i <2000 else 0.0 for i in range(self.N)]))

        self.a, self.Rs, self.Rh = solve_for_driving_signal(h, s, self.bh, self.bs)

        self.alpha = torch.round(self.bs/self.bh,decimals=5)
        
    def forward(self):
        '''
        Forward pass to compute the modeled hard and soft lightcurves from the driving signal and transfer function parameters.
        -----------
        returns:
        h_model : array-like
            Modeled hard lightcurve
        s_model : array-like
            Modeled soft lightcurve
        -----------
        '''

        h_model = torch.zeros(self.N)
        s_model = torch.zeros(self.N)

        for i in np.arange(self.N):
            K_ij = torch.flip(self.Kernel[:i+1], [0])
            a_j = self.a[:i+1]
            # h_model[i] = self.bh * self.a[i] + self.Rh * torch.dot(K_ij, a_j)
            # s_model[i] = self.bs * self.a[i] + self.Rs * torch.dot(K_ij, a_j)
            h_model[i] = torch.add(torch.mul(self.bh, self.a[i]), torch.mul(self.Rh, torch.dot(K_ij,a_j)))
            s_model[i] = torch.add(torch.mul(self.bs, self.a[i]), torch.mul(self.Rs, torch.dot(K_ij,a_j)))  

        return h_model, s_model

In [ ]:
h_tensor = torch.tensor(h, dtype=torch.float32)
s_tensor = torch.tensor(s, dtype=torch.float32)
h_tensor = h_tensor.to(DEVICE)
s_tensor = s_tensor.to(DEVICE)

bH0 = 19./30.
bS0 = 7./3.
bH = 1.00 * bH0
bS = 1.00 * bS0

model = REV_Model(h_tensor,s_tensor,bs=bS,bh=bH)
model = model.to(DEVICE)

Loss = nn.MSELoss()
learning_rate = 1e-2
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

loss_history = []
resume = True
train = True

start_epoch = 0
epochs = 2000

model_name = f"RespSolver-1Process_{bin_size:.0f}binsize_{bH:.2f}bh_{bS:.2f}bs_"
if noisemode !=None:
    model_name += noisemode + "_"
if snr !=None:
    model_name += f"{snr:.1e}snr_"
model_name += f"{learning_rate:.0e}lr"
# TODO model_name += "_" + # You can add Customized label here

scheduler = lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=1e-2, total_iters=int(0.75 * epochs))

if train:
    if resume and os.path.exists(f"Trained_Model/{model_name}.pt"):
        model, optimizer, scheduler, start_epoch, loss_history = load_model(f"{model_name}", model, optimizer, scheduler) 
        print(f"Resuming training from epoch {start_epoch}")
    else:
        print(f"Starting training from scratch: {model_name}")

    model.train()

    for epoch in tqdm(range(start_epoch,epochs)):

        optimizer.zero_grad()

        h_model, s_model = model()
        h_model.to(DEVICE)
        s_model.to(DEVICE)
        
        loss1 = Loss(h_model, h_tensor)
        loss1.backward()

        loss2 = Loss(s_model, s_tensor)
        loss2.backward()

        # Optional: Clamp the parameters to be non-negative (if desired, but not necessary for this problem)
        # for p in model.parameters():
        #     p.data.clamp_(min = 0, max = None)

        optimizer.step()
        scheduler.step()
        loss_history.append(loss1.item()+loss2.item())

        # Checkpoint will be saved and overwrite the previous checkpoint every 100 epochs.
        if (epoch+1) % 100.0 == 0:
            torch.save({
                    'epoch': epoch +1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'loss': loss_history,
                    },
                f"Trained_Model/{model_name}.pt")
            
else:
    model, optimizer, scheduler, start_epoch, loss_history = load_model(f"{model_name}",model,optimizer,scheduler)
    print("Load trained model")

In [ ]:
# Export predicted Response

# Note: The predicted response is the learned transfer function (Kernel) from the model, which is normalized such that the area under the curve is 1. The predicted response is saved as a CSV file in the "Predicted_Response" directory with a name that includes the parameters used for training.
solution = model.Kernel
solution = solution.detach().numpy()

# Normalize the solution such that the area under the curve in specific temporal range is 1. 
# The area under the curve is approximated using the trapezoidal rule with a step size of bin_size (which is 5.0 in this case).
solution = solution/np.trapz(solution[:int(500/bin_size)],dx=bin_size)

os.makedirs('Predicted_Response/', exist_ok=True)
out_file = 'Predicted_Response/' + GTs[1].split("/")[1].split(".")[0] + f"_{100*bH/bH0:.1f}bh_{100*bS/bS0:.1f}bs"
if snr !=None:
    out_file += f"_{snr:.1e}snr"
out_file += ".csv"
out_data = pd.DataFrame({"Time" : T, "Response" :solution})
try:
    out_data.to_csv(out_file)
    print(f"Predicted Response saved to {out_file}")
except:
    raise Exception(f"Failed to save Predicted Response. Check if the directory Predicted_Response exists")